# MSU FRUIT · Demo: fine-tune SAM 2.1 on grape clusters and score it

This notebook previews the whole pipeline on public sample data, end to end: **clips → labels → score pretrained SAM 2.1 → fine-tune it → score again → look at the result**. It is the short version of `sam2_pipeline/notebooks/MSU_FRUIT_SAM2_pipeline.ipynb`, which does the same on our own vineyard video (with uploading and labelling).

- **Runs on a laptop CPU in about 5–10 minutes** (seconds per step on a GPU). It uses the smallest SAM 2.1 model (*tiny*) so it finishes without a GPU.
- **First run downloads about 170 MB**: Meta's SAM 2 code, the SAM 2.1 tiny weights, and four grape photos with hand-drawn cluster outlines (Embrapa WGISD, CC BY-NC 4.0). They are cached in `~/.fruit_sam2/`.
- **Everything it makes goes to `data/demo_project/`** in this repository (git ignores it). Delete that folder to start over.

Setup instructions: `INSTALL.md`. Run the cells top to bottom (**Run ▸ Run All Cells**).

## 1. Load the pipeline
`sam2_pipeline/fruit_sam2.py` holds every tool: clips and labels, SAM 2.1, scoring, training. The HPCC jobs and the full notebook run the same code.

In [ ]:
import os, sys, subprocess
from pathlib import Path

# Opened in Google Colab straight from GitHub? Then fetch the rest of the repository next to it first.
BRANCH = "Saitej"          # the branch holding sam2_pipeline/ (change to "main" once it is merged)
if not Path("sam2_pipeline/fruit_sam2.py").exists() and "google.colab" in sys.modules:
    subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", BRANCH,
                    "https://github.com/moehocbaidi/Capstone-MSU_Fruit.git", "/content/Capstone-MSU_Fruit"], check=True)
    os.chdir("/content/Capstone-MSU_Fruit")
assert Path("sam2_pipeline/fruit_sam2.py").exists(), "Start Jupyter from the repository folder (see INSTALL.md)."

sys.path.insert(0, str(Path("sam2_pipeline").resolve()))
import fruit_sam2 as F        # first run: downloads Meta's SAM 2 code (a few MB)

F.SAM2_SIZE = "tiny"          # the smallest SAM 2.1 (39M parameters), so the demo runs on a laptop
if F.DEVICE == "cpu":
    F.SAM2_IMAGE_SIZE = 512   # faster on a CPU, a little less accurate; GPUs use SAM 2's full 1024
F.use_project("data/demo_project")
print("running on", F.GPU_NAME or "CPU", "· project folder:", F.ROOT)

## 2. Sample data: four grape photos, turned into short clips with exact labels
Each photo is turned into a 12-frame clip by sliding a window across it, so the "video" comes with hand-drawn ground truth. Labels are images where each pixel holds a cluster number (0 = background), one PNG per frame. Our own videos and labels use the same format. Three clips are for training and one is held out for scoring.

In [ ]:
clips = F.make_demo_clips(["CDY_2017", "CDY_2019", "SVB_1935", "SVB_1938"])
VAL = "demo_CDY_2019_t0000000"                  # the held-out clip: never trained on, only scored
for c in clips:
    F.set_split(c, "val" if c == VAL else "train")
F.dataset_report(show_examples=False)
F.show_clip(VAL, title="Held-out clip with its true labels (each colour is one grape cluster)")

## 3. Score pretrained SAM 2.1
SAM 2 does not find grapes by itself; it segments and **follows** what it is pointed at. Here it is given one box per cluster, on the first frame where the cluster is well in view, and must track it through the clip. That is the job it does in the full pipeline, where a detector supplies the boxes.

**J&F** is the standard video-segmentation score: the average of **J** (mask overlap, IoU) and **F** (how well the outlines agree), from 0 to 100, over the frames after the hint.

In [ ]:
PRETRAINED = "SAM 2.1 tiny · pretrained · box"
F.clear_leaderboard()
board = F.benchmark({PRETRAINED: F.SAM2Prompted("tiny", prompt="box")})

## 4. Fine-tune SAM 2.1 on the three training clips
This runs **Meta's own SAM 2 training code** with the official fine-tuning recipe, scaled down to one small device. The demo does only 20 steps, enough to show the machinery working (about 1–2 minutes on a CPU). Real runs use thousands of steps on a GPU (Colab, or a batch job on the MSU HPCC). The result is a normal SAM 2.1 checkpoint in `data/demo_project/models/`.

The loss plot jumps around: each step trains on a different random clip with a different kind of hint (a mask, a box or clicks), and some of those are much easier than others. Twenty steps show no trend. The scores in the next step are what to judge a model by.

In [ ]:
ckpt = F.finetune("demo_run", size="tiny", steps=20)
F.plot_training("demo_run")

## 5. Score the fine-tuned model on the same clip, with the same hints
Both rows of the table are scored by the same function on the same held-out clip. This is the comparison the full pipeline makes for any model: other SAM 2.1 sizes, YOLO or YOLOE, SAM 3.1, or your own.

In [ ]:
FINETUNED = "SAM 2.1 tiny · fine-tuned (demo_run) · box"
board = F.benchmark({FINETUNED: F.SAM2Prompted(ckpt=ckpt, prompt="box")})

## 6. Look at it
Top row: the true labels. Below: what each model drew on the same frames.

In [ ]:
F.compare(VAL, [PRETRAINED, FINETUNED], frames=4, video=False)

## What this shows, and what comes next
- **The pipeline works end to end.** Clips, labels, scoring, Meta's trainer, re-scoring and comparison all ran on this computer.
- **The scores are not a result yet.** Twenty training steps on three sample clips cannot be expected to beat pretrained SAM 2.1; a gap of a point or two either way is noise. The real comparison runs on our Riesling video, with thousands of steps and enough held-out clips.
- **Next, with real data:** open `sam2_pipeline/notebooks/MSU_FRUIT_SAM2_pipeline.ipynb` in Google Colab (GPU), or use the HPCC scripts in `sam2_pipeline/hpcc/`. Upload videos, label them (one box per cluster; SAM 2.1 does the outlines), fine-tune, and score every model. `sam2_pipeline/README.md` walks through it.